# MedStream - Diabetes Prediction Model

Trains classification models on records queried directly from the DuckDB analytical warehouse (`patients_analytics` table):
1. Loads data from `medstream_warehouse.duckdb`.
2. One-hot encodes remaining categorical fields.
3. Splits records into training (80%) and testing (20%) sets.
4. Trains a Logistic Regression baseline and a Random Forest Classifier.
5. Evaluates accuracy, precision, recall, and ROC-AUC score.
6. Computes feature importance to identify key clinical risk factors.
7. Exports the trained model to `models/diabetes_model.pkl`.

### 1. Load Data from Warehouse

In [ ]:
import duckdb
import pandas as pd
import numpy as np

# query data directly from warehouse table
con = duckdb.connect('medstream_warehouse.duckdb')
df = con.execute("SELECT * FROM patients_analytics").df()
con.close()

print(f"Loaded {len(df):,} patient records from warehouse")
df.head()

### 2. Feature Encoding & Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# separate target and features
X = df.drop(columns=['diabetes'])
y = df['diabetes']

# one-hot encode categorical features (gender, smoking_history)
X = pd.get_dummies(X, drop_first=True)

# 80% train, 20% test with stratified sampling
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]:,} samples")
print(f"Testing set:  {X_test.shape[0]:,} samples")
print(f"Feature count: {X_train.shape[1]}")

### 3. Model 1: Logistic Regression (Baseline)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
print(f"Logistic Regression Accuracy: {accuracy_score(y_test, y_pred_lr) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, digits=3))

### 4. Model 2: Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# train random forest with balanced tree depth
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print(f"Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, digits=3))

### 5. Feature Importance Analysis
Identify which patient features drive the model's predictions most strongly.

In [ ]:
import matplotlib.pyplot as plt

importance = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
importance.plot(kind='barh', color='#4C72B0')
plt.title('Random Forest Feature Importance')
plt.xlabel('Importance Score')
plt.show()

print("Top 3 predictors:")
print(importance.tail(3)[::-1].round(4))

### 6. Export Trained Model

In [ ]:
import os
import joblib

os.makedirs("models", exist_ok=True)
model_path = "models/diabetes_model.pkl"

joblib.dump(rf, model_path)
print(f"Model exported successfully to {model_path}")